# Assignment 11 — KV-Cache Analysis (100 points)

This assignment mirrors **USAAIO 2025 Round 2 Problem 2, Part 14** and extends it. You will analyze KV-cache memory requirements for MHA, GQA, and MLA, implement caching mechanisms, and understand the practical implications for large language model deployment.

**Notation:**
- $D$: model dimension, $H$: heads, $G$: GQA groups, $r$: MLA rank
- $D_{qk}$: per-head query/key dim, $D_v$: per-head value dim
- $N$: number of layers, $L$: sequence length
- FP16: 2 bytes per value, FP32: 4 bytes per value

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

**WARNING**: You may only use `torch`, `torch.nn`, `torch.nn.functional`, and `numpy`. No other imports are allowed.

## Part 1 (10 points, non-coding task)

**Why KV-cache?**

During autoregressive generation, at step $t$ the model generates token $t$ conditioned on tokens $1, \ldots, t-1$.

1. Without caching, how many times must we compute the key $K_j$ for position $j$ during generation of an $L$-token sequence? (3 points)
2. With KV-cache, how many times? (2 points)
3. What is the computational savings (ratio)? Express as a function of $L$. (3 points)
4. What is the memory cost of the cache? Why is this a trade-off? (2 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 2 (15 points, non-coding task)

**Cache size formulas.**

Derive the KV-cache size per position per layer for each architecture:

1. **MHA**: Each of $H$ heads stores a key vector ($D_{qk}$ values) and value vector ($D_v$ values). Total per position per layer? (3 points)
2. **GQA** with $G$ groups: Each of $G$ groups stores one key and one value. Total? (3 points)
3. **MQA** ($G = 1$): Special case of GQA. Total? (2 points)
4. **MLA** with rank $r$: Only the compressed $C = xW^{DKV} \in \mathbb{R}^r$ is cached. Total? (3 points)
5. Under what condition on $r$ is MLA more memory-efficient than GQA? Derive the inequality. (4 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 3 (15 points, coding task)

Implement a cache size calculator.

```python
def kv_cache_size(
    arch: str,       # 'mha', 'gqa', 'mqa', 'mla'
    D: int,          # model dimension
    H: int,          # heads
    D_qk: int,       # per-head key dim
    D_v: int,        # per-head value dim
    N: int,          # layers
    L: int,          # sequence length
    G: int = None,   # GQA groups
    r: int = None,   # MLA rank
    dtype_bytes: int = 2  # FP16 default
) -> dict:
    """Returns dict with per_position_per_layer, per_layer, total (all in bytes)."""
```

Test with $D=4096, H=32, D_{qk}=D_v=128, N=80, L=8192$:
- MHA, GQA ($G=8$), GQA ($G=4$), MQA ($G=1$), MLA ($r=512$)
- Print a table with results in GB.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 4 (15 points, coding task)

**Implement MHA with KV-cache.**

Implement `MyMHAWithCache` that supports incremental decoding:

```python
class MyMHAWithCache(nn.Module):
    def forward(self, X, kv_cache=None):
        """
        X: (B, L_new, D) — new tokens only
        kv_cache: tuple (K_prev, V_prev) or None
            K_prev: (B, H, L_prev, D_qk)
            V_prev: (B, H, L_prev, D_v)
        Returns: output (B, L_new, D), new_kv_cache
        """
```

When `kv_cache` is provided:
- Compute Q from new tokens only
- Compute K, V from new tokens, concatenate with cached K, V
- Compute attention using full K, V
- Return updated cache

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 5 (10 points, coding task)

Verify that cached and non-cached MHA produce the same output.

1. Create `MyMHAWithCache` with $D=32, H=4, D_{qk}=D_v=8$.
2. Create input $X$ of shape $(1, 5, 32)$ (5 tokens).
3. **Non-cached**: pass all 5 tokens at once. Get output for position 5.
4. **Cached**: pass tokens one at a time (5 steps), using cache from previous step.
5. Assert the output at step 5 matches the non-cached output at position 5.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 6 (10 points, coding task)

**Implement MLA with compressed cache.**

Implement `MyMLAWithCache` that caches only the compressed $C$:

```python
class MyMLAWithCache(nn.Module):
    def forward(self, X, c_cache=None):
        """
        X: (B, L_new, D)
        c_cache: (B, L_prev, r) or None
        Returns: output (B, L_new, D), new_c_cache
        """
```

Key difference from MHA cache: instead of caching K and V (2HD_qk values per position), cache C (r values per position).

At each step:
1. Compute Q from new tokens
2. Compute C from new tokens, concatenate with cached C
3. Compute K = C @ W_UK, V = C @ W_UV from full C
4. Attention and output

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 7 (10 points, coding task)

Compare cache sizes empirically.

1. Create both `MyMHAWithCache` ($D=64, H=4, D_{qk}=D_v=16$) and `MyMLAWithCache` ($D=64, H=4, D_{qk}=D_v=16, r=8$).
2. Run both through 20 steps of autoregressive generation.
3. After each step, record the cache size in bytes (`.numel() * element_size()`).
4. Print a table comparing MHA cache vs. MLA cache at each step.
5. Compute the compression ratio at step 20.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 8 (15 points, non-coding task)

**Real-world deployment analysis.**

Consider deploying a model with $D=8192, H=64, D_{qk}=D_v=128, N=80$ layers on an 8-GPU cluster (80 GB per GPU, 640 GB total).

Model weights: 70B parameters in FP16 = 140 GB.
Available for cache: 640 - 140 = 500 GB.

1. For MHA: maximum batch_size $\times$ sequence_length product? (3 points)
2. For GQA ($G=8$): same calculation. (3 points)
3. For MLA ($r=512$): same calculation. (3 points)
4. If we want batch_size = 256 and sequence_length = 4096:
   - Which architectures can support this? (3 points)
   - For those that can't, what is the maximum batch size? (3 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """